In [2]:
#apply StandardScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")
print("Features shape (scaled version):", X.shape)

Features shape (scaled version): (2930, 172)


In [3]:
#Apply StandardScaler
#Now X_scaled contains all features standardized (mean = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape (scaled version):", X_scaled.shape)

Features shape (scaled version): (2930, 172)


In [4]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [5]:
#K-Means on Scaled Data
start_time = time.time()
kmean_scaled = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    kmean_scaled.append({"algorithm": "KMeans", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 5.107634544372559 seconds
K-Means runtime: 5.1076 seconds


In [6]:
#Gaussian Mixture (GMM)on Scaled Data
start_time = time.time()
gmm_scaled = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    gmm_scaled.append({"algorithm": "GMM", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 154.71251726150513 seconds
GMM runtime: 154.7125 seconds


In [7]:
#Agglomerative Clustering on Scaled Data
start_time = time.time()
agg_scaled = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    agg_scaled.append({"algorithm": "Agglomerative", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 4.137087821960449 seconds
Agglomerative runtime: 4.1371 seconds


In [8]:
#Spectral Clustering on Scaled Data
start_time = time.time()
spec_scaled = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    spec_scaled.append({"algorithm": "Spectral", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 6.065194368362427 seconds
Spectral runtime: 6.0652 seconds


In [ ]:
#DBSCAN on Scaled Data
start_time = time.time()
dbscan_scaled = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_scaled)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 :  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_scaled[mask], labels[mask])
        dbscan_scaled.append({"algorithm": "DBSCAN", "preprocessing": "Scaled", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Dbscan runtime: {runtime:.4f} seconds")

Runtime: 0.1468522548675537 seconds
Dbscan runtime: 0.1469 seconds


In [10]:

from sklearn.cluster import Birch
start_time = time.time()
birch_scaled = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_scaled)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_scaled, labels)
        birch_scaled.append({
            "algorithm": "BIRCH",
            "preprocessing": "Scaled",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Runtime: 5.505797624588013 seconds
Birch runtime: 5.5058 seconds


In [11]:
from sklearn.cluster import OPTICS
start_time = time.time()
optics_scaled = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_scaled)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_scaled, labels)
        optics_scaled.append({
            "algorithm": "OPTICS",
            "preprocessing": "Scaled",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 55.044068336486816 seconds
Optics runtime: 55.0441 seconds


In [12]:
import csv


ames_results_scaled = (kmean_scaled + gmm_scaled + agg_scaled + spec_scaled + dbscan_scaled+birch_scaled + optics_scaled)
# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/ames_data/ames_scaled.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_scaled)